# Lab 4: Dormant Black Holes in Gaia DR3 from Summary Statistics

**ASTR3986: Astrostatistics**

Most stellar-mass black holes don't accrete, so they are invisible in X-rays. When one orbits a
normal star, though, it pulls the star around, and Gaia sees that in two numbers it publishes for
every bright star:

- **RUWE**: how badly a single-star (5-parameter) model fits the star's positions. Orbital wobble inflates it.
- **`radial_velocity_error`**: built from the scatter of the star's epoch radial velocities. Orbital motion inflates it.

Following Müller-Horn et al. (A&A 709, A62, 2026), we write a **forward model** that predicts both
numbers for a star with a dark companion of mass $M_\mathrm{CO}$ and period $P$, then use **MCMC** to
get the posterior on $(P, M_\mathrm{CO})$ for each star. Stars whose posterior-mean
$M_\mathrm{CO} \geq 3\,M_\odot$ (too heavy for a white dwarf or neutron star) are black-hole candidates.

**Part A (Tasks 1–9)** replicates the paper. **Part B (Tasks 10–13)** applies the pipeline to stars
the paper did not search.

| Paper funnel | Stars |
|---|---|
| Giants with epoch RVs (step 1) | 3,375,482 |
| RUWE > 1.4 and RV amplitude > 20 km/s (step 2) | 37,421 |
| Cross-matched to Zhang+2023, Teff < 6000 K (step 3) | 21,028 |
| Posterior-mean $M_\mathrm{CO} \geq 3\,M_\odot$ | 1,410 |
| After contaminant cuts (published version) | 389 |

## Task 0: Setup

**Environment.** `pip install numpy scipy matplotlib astropy astroquery emcee corner joblib`

**gaiamock** (the paper's astrometry simulator) ships prebuilt wheels only for Linux and macOS.
On Windows, pick one:
1. Run this notebook from **WSL2** (PyCharm can use a WSL interpreter), then `pip install gaiamock`; or
2. `conda install -c conda-forge gsl` then `pip install git+https://github.com/kareemelbadry/gaiamock.git` (needs a C compiler); or
3. Set `USE_GAIAMOCK = False` below and implement `predict_ruwe_simple` (pure numpy). This is a
   stated deviation from the paper, and Task 5 measures what it costs you.

gaiamock also needs the scanning-law data (`healpix_scans.zip`, ~1 GB unzipped):
run `gaiamock-download`, or download it in a browser and run `gaiamock-download path/to/healpix_scans.zip`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import lab4_utils as lu
import importlib; importlib.reload(lu)   # pick up edits to lab4_utils without restarting

rng = np.random.default_rng(42)
%matplotlib inline

USE_GAIAMOCK = False          # True once gaiamock + scanning law are installed
N_CORES = 8                   # for the cost estimates and the batch run
DATA_DIR = lu.DATA_DIR
DATA_DIR.mkdir(exist_ok=True)
print("data ->", DATA_DIR)

In [ ]:
# Forward-model switch: everything downstream calls predictors["ruwe"] / predictors["sigma_rv"],
# so swapping gaiamock <-> the simple model is a one-line change here.
if USE_GAIAMOCK:
    from gaiamock import gaiamock as gm
    c_funcs = gm.read_in_C_functions()
    ruwe_fn = lambda theta, star: lu.predict_ruwe_gaiamock(theta, star, c_funcs)
else:
    ruwe_fn = lambda theta, star: lu.predict_ruwe_simple(theta, star, rng)

predictors = {
    "ruwe": ruwe_fn,
    "sigma_rv": lambda theta, star: lu.predict_sigma_rv(theta, star, rng),
}

---
# Part A: Replicating the paper

## Task 1: Ingest the parent sample efficiently

Gaia DR3 has ~1.8 billion sources, and we only want ~40,000 of them. Instead of downloading and
then filtering, we send the paper's cuts to the archive as ADQL, so the **filtering happens
server-side** and we download only the stars that pass. We also join
`gaiadr3.astrophysical_parameters` in the same query to get FLAME masses and GSP-Phot $T_\mathrm{eff}$
(our substitute for the paper's Zhang+2023 cross-match). The result is cached in `data/`, so
re-running the notebook doesn't hit the archive again.

Absolute magnitude from parallax in mas: $M_G = G + 5\log_{10}(\varpi/100)$.

In [ ]:
query = lu.build_parent_query()          # paper's step-1 + step-2 cuts
print(query)

In [ ]:
parent = lu.fetch_gaia(query, DATA_DIR / "parent_step2.ecsv")
print(len(parent), "stars;  paper step 2:", lu.PAPER_FUNNEL["step2_ruwe_rvamp"])

## Task 2: Reproduce the selection funnel

Count step 1 with a `COUNT(*)` query (no download), then apply step 3 locally. Put our counts next
to the paper's and explain any difference. (Ours uses GSP-Phot $T_\mathrm{eff}$, not Zhang+2023, so
step 3 will not match exactly. Say by how much.)

In [ ]:
n_step1 = lu.fetch_gaia(lu.build_count_query(include_step2=False), DATA_DIR / "count_step1.ecsv")["n"][0]

# Step 3 (our version): Teff < 6000 K and a usable primary mass
# TODO: decide how to treat stars with no teff_gspphot (drop? keep?), and justify it
step3 = parent[(parent["teff_gspphot"] < lu.CUTS_PAPER["teff_max"])]

funnel = {"step1_giants": n_step1, "step2_ruwe_rvamp": len(parent), "step3_teff": len(step3)}
for k, v in funnel.items():
    paper_key = {"step3_teff": "step3_zhang"}.get(k, k)
    print(f"{k:18s} ours = {v:>10,}   paper = {lu.PAPER_FUNNEL[paper_key]:>10,}")

In [ ]:
# CMD of the selected sample: should be the red giant branch / red clump
absG = parent["phot_g_mean_mag"] + 5 * np.log10(parent["parallax"] / 100)
plt.scatter(parent["bp_rp"], absG, s=1, c="k", alpha=0.3)
plt.gca().invert_yaxis()
plt.xlabel("BP - RP"); plt.ylabel(r"$M_G$")
plt.title("Step-2 sample")
plt.show()

## Task 3: Primary masses

The paper **fixes** $M_*$ for each star (it is not an MCMC parameter), using an XGBoost model.
We use Gaia FLAME masses with a fallback for missing values. Implement `lu.estimate_primary_mass`,
then show the mass distribution and the fraction that needed the fallback.

*Why it matters:* $K_*$ and the photocentre orbit both depend on $M_*$, so a mass that is too low
pushes the inferred $M_\mathrm{CO}$ down. Later, in Task 9, check how much $M_\mathrm{CO}$ moves when
$M_*$ changes by ±20%.

In [ ]:
mstar, used_fallback = lu.estimate_primary_mass(step3)
step3["mstar"] = mstar
print(f"fallback used for {used_fallback.mean():.1%} of stars")

plt.hist(mstar, bins=50, color="steelblue")
plt.xlabel(r"$M_*$ [$M_\odot$]"); plt.ylabel("N")
plt.show()

## Task 4: Forward model I: radial velocities

For a dark companion the visible star moves on a Keplerian orbit with

$$K_* = \frac{M_\mathrm{CO}}{(M_*+M_\mathrm{CO})^{2/3}}\left(\frac{2\pi G}{P}\right)^{1/3}\frac{\sin i}{\sqrt{1-e^2}},
\qquad v(t) = v_z + K_*\left[\cos(\omega+\nu(t)) + e\cos\omega\right].$$

Gaia doesn't publish the epoch RVs. It publishes `radial_velocity_error`, which encodes their scatter:

$$\sigma_\mathrm{RV} = 1.07\sqrt{\left(\sqrt{\tfrac{\pi}{2N}}\,\tilde\sigma_\mathrm{RV}\right)^2 + (0.113\ \mathrm{km\,s^{-1}})^2}$$

where $\tilde\sigma_\mathrm{RV}$ is the standard deviation of the $N$ = `rv_nb_transits` simulated epoch RVs.

Implement `solve_kepler`, `true_anomaly`, `semi_amplitude`, `rv_curve`, `rv_epoch_times`, `predict_sigma_rv`.

In [ ]:
# Unit tests: run these before anything else depends on the functions
# 1) Kepler's equation is satisfied
Mgrid = np.linspace(0, 2 * np.pi, 1000)
for e in [0.0, 0.3, 0.9]:
    E = lu.solve_kepler(Mgrid, e)
    assert np.allclose(E - e * np.sin(E), Mgrid, atol=1e-8), e

# 2) Sun-Jupiter semi-amplitude ~ 12.5 m/s
K_jup = lu.semi_amplitude(1.0, 9.55e-4, 4332.6, 0.049, np.pi / 2)
print(f"K(Jupiter) = {K_jup * 1e3:.2f} m/s  (expect ~12.5)")

# 3) Gaia BH1-like system: P = 185.6 d, e = 0.45, M_* ~ 0.93, M_BH ~ 9.6, i = 126.6 deg -> K ~ 67 km/s (published 66.7)
print(f"K(BH1-like) = {lu.semi_amplitude(0.93, 9.62, 185.6, 0.45, np.deg2rad(126.6)):.1f} km/s")

In [ ]:
# How sigma_RV grows with companion mass and period, for one representative giant
star0 = lu.row_to_star(step3[0], mstar_col="mstar")
Ps = np.logspace(1, 4, 60)
masses = [0.5, 1.5, 3, 10]
for m in masses:
    sig = [lu.predict_sigma_rv(np.array([np.log10(P), np.log10(m), 0.5, 0.3, 1.0, 1.0, 0.2]), star0, rng)
           for P in Ps]
    plt.loglog(Ps, sig, label=fr"$M_{{CO}}={m}\,M_\odot$")
plt.axhline(star0["radial_velocity_error"], color="k", ls="--", label="this star (observed)")
plt.xlabel("P [days]"); plt.ylabel(r"predicted $\sigma_\mathrm{RV}$ [km/s]"); plt.legend()
plt.show()

## Task 5: Forward model II: RUWE

The photocentre of a star with a dark companion traces an ellipse of angular semi-major axis
$a_* = a_\mathrm{tot}\,\frac{M_\mathrm{CO}}{M_*+M_\mathrm{CO}}\,\varpi$. Gaia fits a 5-parameter single-star
model through that wobble, and the misfit shows up as

$$\mathrm{RUWE} \approx \sqrt{\chi^2/\nu}, \qquad \nu = \texttt{astrometric\_n\_good\_obs\_al} - 5.$$

Implement `predict_ruwe_gaiamock` or `predict_ruwe_simple` (or both).

**Calibration check:** the paper's likelihood assumes the model predicts ln RUWE to within
$\varepsilon_\mathrm{ruwe} = 0.14$. Repeat one prediction ~200 times with different noise draws and
report the scatter of ln RUWE. If you use the simple model, compare it to gaiamock's scatter
(or to 0.14) and say whether you need a larger $\varepsilon$.

In [ ]:
theta_test = np.array([np.log10(500), np.log10(10), 0.5, 0.3, 1.0, 1.0, 0.2])   # 10 Msun BH, P = 500 d
print("photocentre a* =", lu.photocenter_semimajor_axis_mas(500, star0["mstar"], 10, star0["parallax"]), "mas")

ruwe_draws = np.array([predictors["ruwe"](theta_test, star0) for _ in range(200)])
print(f"median RUWE = {np.median(ruwe_draws):.2f},  std(ln RUWE) = {np.std(np.log(ruwe_draws)):.3f}"
      f"  (paper eps = {lu.EPS_RUWE})")

In [ ]:
# RUWE vs period for several companion masses (same layout as the sigma_RV plot)
for m in masses:
    r = [predictors["ruwe"](np.array([np.log10(P), np.log10(m), 0.5, 0.3, 1.0, 1.0, 0.2]), star0) for P in Ps]
    plt.loglog(Ps, r, label=fr"$M_{{CO}}={m}\,M_\odot$")
plt.axhline(star0["ruwe"], color="k", ls="--", label="this star (observed)")
plt.xlabel("P [days]"); plt.ylabel("predicted RUWE"); plt.legend()
plt.show()

## Task 6: Likelihood and priors

$$\ln\mathcal{L} = -\tfrac12\ln(2\pi\varepsilon_r^2) - \tfrac12\left(\frac{\ln\mathrm{ruwe}_\mathrm{obs}-\ln\mathrm{ruwe}_\mathrm{pred}}{\varepsilon_r}\right)^2
- \tfrac12\ln(2\pi\varepsilon_s^2) - \tfrac12\left(\frac{\ln\sigma_\mathrm{obs}-\ln\sigma_\mathrm{pred}}{\varepsilon_s}\right)^2$$

| Parameter | Prior |
|---|---|
| $\log_{10}(P/\mathrm{d})$ | U[1, 4] |
| $\log_{10}(M_\mathrm{CO}/M_\odot)$ | U[log 0.05, log 30] |
| $\cos i$ | U(0, 1] |
| $e$ | U[0, 0.95] |
| $\Omega,\ \omega$ | U[0, 2π) |
| $\phi_p = T_p/P$ | U[0, 1) |

Why log space? The model's scatter is roughly a constant *fraction*, so it's Gaussian in the log.
Note one consequence: the forward model is **stochastic** (noise draws), so the likelihood is noisy.
Think about what that does to an MCMC, and whether to fix the seed per star.

Implement `log_prior`, `sample_prior`, `log_likelihood`, `log_posterior`.

In [ ]:
# Tests: prior draws are inside the prior; the posterior is finite there and -inf outside
draws = lu.sample_prior(1000, rng)
assert np.all(np.isfinite([lu.log_prior(t) for t in draws]))
assert lu.log_prior(np.array([5.0, 0, 0.5, 0.3, 1, 1, 0.2])) == -np.inf     # logP = 5 is outside
print("log-post at theta_test:", lu.log_posterior(theta_test, star0, predictors))

In [ ]:
# 2-D log-likelihood map over (logP, logMco) for one star, marginalising nothing (other params fixed).
# Shows the shape the sampler has to explore: expect a banana / multiple ridges in P.
logP_grid = np.linspace(*lu.LOGP_RANGE, 40)
logM_grid = np.linspace(*lu.LOGMCO_RANGE, 40)
LL = np.array([[lu.log_likelihood(np.array([lp, lm, 0.5, 0.3, 1, 1, 0.2]), star0, predictors)
                for lp in logP_grid] for lm in logM_grid])
plt.pcolormesh(logP_grid, logM_grid, LL, shading="auto", vmin=LL.max() - 20)
plt.colorbar(label="ln L"); plt.xlabel(r"$\log_{10} P$ [d]"); plt.ylabel(r"$\log_{10} M_{CO}$")
plt.show()

## Task 7: MCMC: emcee (the paper's sampler) vs our own Metropolis-Hastings

Paper settings: **32 walkers, 2500 steps, 500 burn-in**, about 10 s per star per core.

1. Run `lu.run_emcee` on one star and plot traces for $\log P$ and $\log M_\mathrm{CO}$.
2. Run `lu.run_metropolis` (Lab 3's sampler, adapted) on the same star. Compare acceptance
   fraction, autocorrelation time and posterior.
3. Convergence: the paper checks (a) that different initialisations agree, (b) stability between
   the 2nd and final thirds of the chain, and (c) that a 10× longer chain gives the same answer.
   Do (a) and (b) on at least one star, and add the integrated autocorrelation time.

In [ ]:
import time
t0 = time.time()
chain, logp = lu.run_emcee(star0, predictors, rng)
print(f"emcee: {time.time() - t0:.1f} s for one star (paper ~ {lu.SEC_PER_STAR:.0f} s)")

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for ax, k, name in zip(axes, [0, 1], [r"$\log_{10}P$", r"$\log_{10}M_{CO}$"]):
    ax.plot(chain[:, :, k], color="k", alpha=0.15, lw=0.5)
    ax.axvline(lu.N_BURN, color="r"); ax.set_ylabel(name)
axes[-1].set_xlabel("step")
plt.show()

In [ ]:
summary = lu.summarize_posterior(chain)
print(summary)
print(lu.convergence_checks(chain))

import corner
flat = chain[lu.N_BURN:].reshape(-1, lu.NDIM)
corner.corner(flat[:, :4], labels=lu.PARAM_NAMES[:4], show_titles=True)
plt.show()

In [ ]:
# Our own MH sampler on the same star, for comparison
theta_start = lu.sample_prior(1, rng)[0]
jump = np.array([0.1, 0.1, 0.05, 0.05, 0.3, 0.3, 0.05])   # TODO: tune as in Lab 3
chain_mh, lp_mh, acc_mh = lu.run_metropolis(theta_start, jump, 20000, star0, predictors, rng)
print(f"MH acceptance = {acc_mh:.3f}")

plt.hist(flat[:, 1], bins=50, density=True, alpha=0.6, label="emcee")
plt.hist(chain_mh[5000:, 1], bins=50, density=True, alpha=0.6, label="our MH")
plt.xlabel(r"$\log_{10} M_{CO}$"); plt.legend()
plt.show()

## Task 8: Injection-recovery test

Before trusting anything on real data: take real host giants, give them a fake black hole of
known mass and period, simulate their RUWE and $\sigma_\mathrm{RV}$, run the full inference, and
check that we get the truth back. The paper uses 5, 10 and 20 $M_\odot$ and periods from 10 to
30,000 d. They report ≳80% completeness for 10–20 $M_\odot$ at 100–2000 d, falling at long
periods because Gaia's 34-month baseline only covers part of the orbit.

Lab-sized version: 3 masses × 6 periods × ~10 hosts. Use `lu.estimate_cost` before you launch it.

In [ ]:
inj_masses = [5, 10, 20]
inj_periods = np.logspace(1.5, 4, 6)
n_hosts = 10
lu.estimate_cost(len(inj_masses) * len(inj_periods) * n_hosts, n_cores=N_CORES)

In [ ]:
results = []   # rows: (true_m, true_P, Mco_mean, p_gt_threshold)
hosts = step3[rng.choice(len(step3), n_hosts, replace=False)]
for m in inj_masses:
    for P in inj_periods:
        for h in hosts:
            mock, theta_true = lu.make_mock_star(lu.row_to_star(h, mstar_col="mstar"), m, P, rng, predictors)
            ch, _ = lu.run_emcee(mock, predictors, rng)
            s = lu.summarize_posterior(ch)
            results.append((m, P, s["Mco_mean"], s["p_gt_threshold"]))
results = np.array(results)
np.save(DATA_DIR / "injection_results.npy", results)

In [ ]:
# Completeness = fraction recovered with Mco_mean >= 3 Msun, per (mass, period)
for m in inj_masses:
    sel = results[:, 0] == m
    comp = [np.mean(results[sel & (results[:, 1] == P), 2] >= lu.M_CO_THRESHOLD) for P in inj_periods]
    plt.semilogx(inj_periods, comp, "o-", label=fr"{m} $M_\odot$")
plt.xlabel("P [days]"); plt.ylabel("completeness"); plt.ylim(0, 1.05); plt.legend()
plt.show()

## Task 9: Validation on real systems

1. **Known black holes.** Fetch Gaia BH2 and BH3 and run the pipeline. Both should come out with
   $M_\mathrm{CO} \gg 3\,M_\odot$. (BH1 is a Sun-like dwarf that fails the giant cut, so run it as a
   stress test only. BH2 was removed in the paper by the "already has an NSS orbit" cut.)
2. **The paper's catalogue.** The paper's 21,028-star catalogue is on Zenodo (find the link in
   its data-availability section). Run our pipeline on a random ~200 of those stars and plot our
   $M_\mathrm{CO}$ against theirs. This is the main replication result.
3. **Sensitivity to $M_*$:** rerun a few stars with $M_*$ ±20%.

In [ ]:
known = lu.fetch_gaia(lu.build_id_query(lu.KNOWN_BHS.values()), DATA_DIR / "known_bhs.ecsv")
id_to_name = {v: k for k, v in lu.KNOWN_BHS.items()}
for row in known:
    star = lu.row_to_star(row)          # uses mass_flame; TODO: override with literature M_* if missing
    ch, _ = lu.run_emcee(star, predictors, rng)
    s = lu.summarize_posterior(ch)
    print(f"{id_to_name[row['source_id']]:9s}  ruwe={row['ruwe']:.2f}  "
          f"sigRV={row['radial_velocity_error']:.2f}  ->  Mco_mean={s['Mco_mean']:.1f}  "
          f"P(M>3)={s['p_gt_threshold']:.2f}")

In [ ]:
# Replication against the published catalogue (download it from Zenodo into data/ first)
from astropy.table import Table
paper_cat = Table.read(DATA_DIR / "paper_catalogue.csv")      # TODO: match the real file name/columns
sub = paper_cat[rng.choice(len(paper_cat), 200, replace=False)]
# TODO: fetch these source_ids with lu.build_id_query, run lu.run_batch, then join on source_id
# and scatter-plot our Mco_mean against theirs with a 1:1 line

---
# Part B: Searching for new candidates

The paper's cuts leave several groups of stars nobody has run this inference on. Pick one or more:

| Search sample | Why there could be black holes there | ADQL change |
|---|---|---|
| **B1: just below the cuts** | low-inclination or long-period systems have smaller RUWE / RV amplitude | `ruwe_min=1.25`, `rv_amp_min=10`, minus the step-2 sample |
| **B2: stars with DR3 orbits** | the paper dropped all 450 of them (that's how it lost BH2) | `extra_where="g.non_single_star > 0"` |
| **B3: DR4 (expected Dec 2026)** | 66 months of epoch astrometry; same model, longer baseline | rerun with `data_release="dr4"` |

Whatever comes out is a **candidate list**. Confirming any of them needs spectroscopic follow-up.

## Task 10: Ingest the search sample

Same server-side approach as Task 1. If a query times out, split it into chunks with
`extra_where` on `source_id` ranges (source_id encodes HEALPix level-12 sky position, so ranges
are sky patches) and concatenate the cached pieces.

In [ ]:
# B1 example: relaxed cuts, then remove anything already in the paper's step-2 sample
relaxed = lu.fetch_gaia(lu.build_parent_query(cuts=dict(ruwe_min=1.25, rv_amp_min=10.0)),
                        DATA_DIR / "search_B1_relaxed.ecsv")
new_mask = ~np.isin(relaxed["source_id"], parent["source_id"])
search = relaxed[new_mask & (relaxed["teff_gspphot"] < lu.CUTS_PAPER["teff_max"])]
print(f"{len(search):,} stars in the B1 search sample")
lu.estimate_cost(len(search), n_cores=N_CORES)

In [ ]:
# Chunked version for big queries (uncomment if the single query times out)
# edges = np.linspace(0, 2**63 - 1, 9).astype(np.int64)    # 8 sky chunks
# pieces = [lu.fetch_gaia(lu.build_parent_query(cuts=..., extra_where=f"g.source_id BETWEEN {a} AND {b}"),
#                         DATA_DIR / f"chunk_{i}.ecsv") for i, (a, b) in enumerate(zip(edges[:-1], edges[1:]))]
# from astropy.table import vstack; relaxed = vstack(pieces)

## Task 11: Batch inference

`lu.run_batch` runs the MCMC over every star in parallel and writes a summary row per star to disk
as it goes. If it crashes or you close the laptop, rerunning picks up where it stopped. Start with
~100 stars to time it, then launch the rest (overnight, or on a cluster).

In [ ]:
search["mstar"], _ = lu.estimate_primary_mass(search)
stars = [lu.row_to_star(r, mstar_col="mstar") for r in search]

out_csv = DATA_DIR / "search_B1_posteriors.csv"
lu.run_batch(stars[:100], predictors, out_csv, n_jobs=N_CORES)    # timing run first

## Task 12: Contaminant filters

Apply the paper's cuts to anything with posterior-mean $M_\mathrm{CO} \geq 3\,M_\odot$:

- **A**: $\sigma_\mathrm{RV}$ inconsistent with `rv_amplitude_robust`
- **B**: unreliable parallax
- **C**: photometric variability larger than ellipsoidal variation can explain
- **D**: already has a DR3 orbit (skip this one for search sample B2)

Report how many stars each cut removes, as the paper does.

Remaining contaminants to discuss: hierarchical triples (an inner binary acting as one "dark",
heavy companion) and luminous companions (our model assumes the companion is dark).

In [ ]:
from astropy.table import Table, join
post = Table.read(out_csv)
cands = join(search, post, keys="source_id")
cands = cands[cands["Mco_mean"] >= lu.M_CO_THRESHOLD]
print(f"{len(cands)} stars with Mco_mean >= {lu.M_CO_THRESHOLD} Msun before cleaning")

flags = {
    "A_sigrv": lu.flag_sigma_rv_outliers(cands, sims=results),
    "B_parallax": lu.flag_parallax_disagreement(cands, plx_alt_col=None),   # TODO: choose an alternative parallax
    "C_photvar": lu.flag_photometric_variables(cands, post),
    "D_nss": lu.flag_has_nss_orbit(cands),
}
for k, f in flags.items():
    print(f"{k:12s} removes {f.sum()}")
clean = cands[~np.any(np.column_stack(list(flags.values())), axis=1)]
print(f"{len(clean)} candidates after cleaning")

## Task 13: Candidates

Plots: (i) $M_\mathrm{CO}$ vs $P$ with error bars, with the paper's candidates overlaid in another
colour; (ii) CMD of the candidates on top of the parent sample; (iii) sky positions. Save the list
to `data/lab4_candidates.csv` with posterior summaries.

In [ ]:
plt.errorbar(clean["P_mean"], clean["Mco_50"],
             yerr=[clean["Mco_50"] - clean["Mco_16"], clean["Mco_84"] - clean["Mco_50"]],
             fmt="o", ms=3, color="k", label="this work (B1)")
plt.axhline(lu.M_CO_THRESHOLD, color="r", ls="--")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("P [days]"); plt.ylabel(r"$M_{CO}$ [$M_\odot$]"); plt.legend()
plt.show()

clean.write(DATA_DIR / "lab4_candidates.csv", overwrite=True)

## Task 14: Summary

- **Replication:** funnel counts vs the paper; our $M_\mathrm{CO}$ vs theirs on the 200-star subset; known BHs recovered?
- **Method:** emcee vs our MH (acceptance, autocorrelation, multimodality in $P$); the cost of our RUWE model if not gaiamock.
- **Completeness:** injection-recovery curve vs the paper's (≳80% at 10–20 $M_\odot$, 100–2000 d).
- **New candidates:** how many, from which search sample, and the main contaminants that could explain them.
- **Deviations from the paper:** FLAME masses instead of XGBoost; GSP-Phot $T_\mathrm{eff}$ instead of Zhang+2023; (if used) simplified RUWE model.